**Imports and configuration**

In [0]:
# ============================================================
# DB_05_Train_Pricing_Anomaly_Model
#
# Purpose:
# - Read DB_04 pricing anomaly feature stores
# - Train an unsupervised Isolation Forest model
# - Use 2025 as an untouched temporal diagnostic period
# - Retrain production candidate using 2023-2025
# - Score 2026 PO-item pricing
# - Log models and metrics to MLflow
# - Persist validation, scoring, and model metadata to OneLake
# ============================================================

from datetime import datetime, timezone

import math
import numpy as np
import pandas as pd

import sklearn

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Temporal design
# ------------------------------------------------------------

WARMUP_YEAR = 2022

DEVELOPMENT_START_YEAR = 2023
DEVELOPMENT_END_YEAR = 2024

TEMPORAL_VALIDATION_YEAR = 2025

SCORING_YEAR = 2026


# ------------------------------------------------------------
# Isolation Forest
# ------------------------------------------------------------

RANDOM_STATE = 20260802

N_ESTIMATORS = 300

MAX_SAMPLES = "auto"

MAX_FEATURES = 1.0

BOOTSTRAP = False


# ------------------------------------------------------------
# Operating threshold
#
# This is a review-capacity threshold, not an assumed
# ground-truth anomaly prevalence.
# ------------------------------------------------------------

ANOMALY_REVIEW_RATE = 0.05

ANOMALY_SCORE_CUTOFF = (
    100.0
    *
    (
        1.0
        -
        ANOMALY_REVIEW_RATE
    )
)


# ------------------------------------------------------------
# Model lifecycle
# ------------------------------------------------------------

MODEL_FAMILY = "Pricing Anomaly Detection"

MODEL_NAME = "IsolationForest"

MODEL_STATUS = "Experimental"


print(
    "DB_05 configuration loaded."
)

print(
    "scikit-learn version:",
    sklearn.__version__
)

print(
    "MLflow version:",
    mlflow.__version__
)

print(
    "Development years:",
    (
        DEVELOPMENT_START_YEAR,
        DEVELOPMENT_END_YEAR
    )
)

print(
    "Untouched temporal diagnostic year:",
    TEMPORAL_VALIDATION_YEAR
)

print(
    "Scoring year:",
    SCORING_YEAR
)

print(
    "Anomaly review rate:",
    f"{ANOMALY_REVIEW_RATE:.2%}"
)

print(
    "Anomaly score cutoff:",
    ANOMALY_SCORE_CUTOFF
)

DB_05 configuration loaded.
scikit-learn version: 1.7.2
MLflow version: 3.12.0
Development years: (2023, 2024)
Untouched temporal diagnostic year: 2025
Scoring year: 2026
Anomaly review rate: 5.00%
Anomaly score cutoff: 95.0


**Load OneLake credentials**

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define DB_04 input and DB_05 output paths**

In [0]:
# ============================================================
# Pricing Anomaly ML paths
# ============================================================

GOLD_LAKEHOUSE_ROOT = (
    "<ABFSS PATH>"
    "<Lakehouse ID>"
)


PRICING_ANOMALY_ML_ROOT = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/pricing_anomaly"
)


# ------------------------------------------------------------
# DB_04 inputs
# ------------------------------------------------------------

TRAINING_FEATURES_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"training_features"
)

SCORING_FEATURES_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"scoring_features"
)


# ------------------------------------------------------------
# DB_05 outputs
# ------------------------------------------------------------

VALIDATION_PREDICTIONS_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"validation_predictions"
)

SCORING_PREDICTIONS_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"scoring_predictions"
)

MODEL_METADATA_PATH = (
    f"{PRICING_ANOMALY_ML_ROOT}/"
    f"model_metadata"
)


print(
    "Training features:",
    TRAINING_FEATURES_PATH
)

print(
    "Scoring features:",
    SCORING_FEATURES_PATH
)

print(
    "Validation predictions:",
    VALIDATION_PREDICTIONS_PATH
)

print(
    "2026 predictions:",
    SCORING_PREDICTIONS_PATH
)

**Read DB_04 feature stores**

In [0]:
# ============================================================
# Read DB_04 pricing feature stores
# ============================================================

pricing_training_df = (
    spark.read
    .format("delta")
    .load(
        TRAINING_FEATURES_PATH
    )
)


pricing_scoring_df = (
    spark.read
    .format("delta")
    .load(
        SCORING_FEATURES_PATH
    )
)


training_row_count = (
    pricing_training_df.count()
)


scoring_row_count = (
    pricing_scoring_df.count()
)


print(
    "DB_04 historical feature rows:",
    f"{training_row_count:,}"
)

print(
    "DB_04 2026 scoring rows:",
    f"{scoring_row_count:,}"
)

DB_04 historical feature rows: 54,023
DB_04 2026 scoring rows: 21,752


**Freeze the DB_04 feature contract**

In [0]:
# ============================================================
# Isolation Forest feature contract
#
# Must remain identical to DB_04.
# ============================================================

PRICING_MODEL_FEATURES = [

    # Historical material pricing
    "LogPriceToMaterialHistoricalRatio",
    "MaterialPriceZScore",
    "AbsoluteMaterialPriceDeviationPct",

    # Supplier-material pricing
    "LogPriceToSupplierMaterialHistoricalRatio",
    "SupplierMaterialPriceZScore",
    "AbsoluteSupplierMaterialPriceDeviationPct",
    "SupplierMaterialHistoricalCVPct",

    # Category pricing
    "LogPriceToCategoryHistoricalRatio",
    "CategoryPriceZScore",
    "AbsoluteCategoryPriceDeviationPct",

    # Governed contract benchmark
    "ContractPriceVariancePct",
    "AbsoluteContractPriceVariancePct",

    # Transaction scale
    "LogUnitPriceEUR",
    "LogQuantity",

    # Benchmark availability
    "HasMaterialHistoryFlag",
    "HasSupplierMaterialHistoryFlag",
    "HasCategoryHistoryFlag",
    "HasContractBenchmarkFlag",
    "BenchmarkCoverageCount"
]


missing_training_features = [
    feature
    for feature
    in PRICING_MODEL_FEATURES
    if feature
    not in pricing_training_df.columns
]


missing_scoring_features = [
    feature
    for feature
    in PRICING_MODEL_FEATURES
    if feature
    not in pricing_scoring_df.columns
]


if missing_training_features:

    raise ValueError(
        "Historical pricing feature store "
        "is missing model features: "
        +
        ", ".join(
            missing_training_features
        )
    )


if missing_scoring_features:

    raise ValueError(
        "2026 pricing feature store "
        "is missing model features: "
        +
        ", ".join(
            missing_scoring_features
        )
    )


print(
    "Pricing feature contract PASSED."
)

print(
    "Model feature count:",
    len(
        PRICING_MODEL_FEATURES
    )
)

Pricing feature contract PASSED.
Model feature count: 19


**Inspect yearly population**

In [0]:
# ============================================================
# Inspect historical pricing population
# ============================================================

display(
    pricing_training_df

    .groupBy(
        "OrderYear"
    )

    .agg(
        F.count("*")
        .alias(
            "POItemCount"
        ),

        F.round(
            F.avg(
                F.col(
                    "RuleBasedExtremePriceFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "RuleBasedExtremePricePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "HasContractBenchmarkFlag"
                )
                .cast("double")
            )
            * 100.0,
            2
        )
        .alias(
            "ContractBenchmarkCoveragePct"
        ),

        F.round(
            F.avg(
                F.col(
                    "BenchmarkCoverageCount"
                )
            ),
            2
        )
        .alias(
            "AverageBenchmarkCoverageCount"
        )
    )

    .orderBy(
        "OrderYear"
    )
)

OrderYear,POItemCount,RuleBasedExtremePricePct,ContractBenchmarkCoveragePct,AverageBenchmarkCoverageCount
2022,12514,11.27,52.44,1.95
2023,10446,8.99,48.05,2.41
2024,12123,8.88,61.56,2.71
2025,18940,8.4,68.23,2.86


**Create temporal populations**

In [0]:
# ============================================================
# Temporal model populations
#
# 2022:
# historical warm-up only
#
# 2023-2024:
# model development
#
# 2025:
# untouched temporal diagnostic
#
# 2026:
# current scoring
# ============================================================

development_df = (
    pricing_training_df

    .filter(
        (
            F.col(
                "OrderYear"
            )
            >=
            DEVELOPMENT_START_YEAR
        )
        &
        (
            F.col(
                "OrderYear"
            )
            <=
            DEVELOPMENT_END_YEAR
        )
    )
)


validation_df = (
    pricing_training_df

    .filter(
        F.col(
            "OrderYear"
        )
        ==
        TEMPORAL_VALIDATION_YEAR
    )
)


production_training_df = (
    pricing_training_df

    .filter(
        (
            F.col(
                "OrderYear"
            )
            >=
            DEVELOPMENT_START_YEAR
        )
        &
        (
            F.col(
                "OrderYear"
            )
            <=
            TEMPORAL_VALIDATION_YEAR
        )
    )
)


development_count = (
    development_df.count()
)

validation_count = (
    validation_df.count()
)

production_training_count = (
    production_training_df.count()
)


print(
    "Development rows (2023-2024):",
    f"{development_count:,}"
)

print(
    "Temporal diagnostic rows (2025):",
    f"{validation_count:,}"
)

print(
    "Production training rows (2023-2025):",
    f"{production_training_count:,}"
)

print(
    "2026 scoring rows:",
    f"{scoring_row_count:,}"
)

Development rows (2023-2024): 22,569
Temporal diagnostic rows (2025): 18,940
Production training rows (2023-2025): 41,509
2026 scoring rows: 21,752


**Convert model matrices to pandas**

In [0]:
# ============================================================
# Convert ML matrices to pandas
# ============================================================

development_pd = (
    development_df

    .select(
        "POItemID",
        *PRICING_MODEL_FEATURES
    )

    .toPandas()
)


validation_pd = (
    validation_df

    .select(
        "POItemID",
        "RuleBasedExtremePriceFlag",
        "PriceComplianceExceptionFlag",
        *PRICING_MODEL_FEATURES
    )

    .toPandas()
)


production_training_pd = (
    production_training_df

    .select(
        "POItemID",
        *PRICING_MODEL_FEATURES
    )

    .toPandas()
)


scoring_pd = (
    pricing_scoring_df

    .select(
        "POItemID",
        "RuleBasedExtremePriceFlag",
        "PriceComplianceExceptionFlag",
        *PRICING_MODEL_FEATURES
    )

    .toPandas()
)


print(
    "Pandas conversion completed."
)

print(
    "Development shape:",
    development_pd.shape
)

print(
    "Validation shape:",
    validation_pd.shape
)

print(
    "Production training shape:",
    production_training_pd.shape
)

print(
    "Scoring shape:",
    scoring_pd.shape
)

Pandas conversion completed.
Development shape: (22569, 20)
Validation shape: (18940, 22)
Production training shape: (41509, 20)
Scoring shape: (21752, 22)


**Numeric matrix preparation helper**

In [0]:
# ============================================================
# Numeric feature preparation
# ============================================================

def prepare_feature_matrix(
    dataframe,
    feature_columns
):

    matrix = (
        dataframe[
            feature_columns
        ]
        .copy()
    )


    for feature_name in feature_columns:

        matrix[
            feature_name
        ] = (
            pd.to_numeric(
                matrix[
                    feature_name
                ],
                errors="coerce"
            )
        )


    matrix = (
        matrix

        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )

        .astype(
            "float64"
        )
    )


    return matrix


X_development = (
    prepare_feature_matrix(
        development_pd,
        PRICING_MODEL_FEATURES
    )
)


X_validation = (
    prepare_feature_matrix(
        validation_pd,
        PRICING_MODEL_FEATURES
    )
)


X_production = (
    prepare_feature_matrix(
        production_training_pd,
        PRICING_MODEL_FEATURES
    )
)


X_scoring = (
    prepare_feature_matrix(
        scoring_pd,
        PRICING_MODEL_FEATURES
    )
)


print(
    "Numeric feature matrices prepared."
)

print(
    "Development matrix:",
    X_development.shape
)

print(
    "Validation matrix:",
    X_validation.shape
)

print(
    "Production matrix:",
    X_production.shape
)

print(
    "Scoring matrix:",
    X_scoring.shape
)

Numeric feature matrices prepared.
Development matrix: (22569, 19)
Validation matrix: (18940, 19)
Production matrix: (41509, 19)
Scoring matrix: (21752, 19)


**Build Isolation Forest pipeline**

The only required transformation here is median imputation. _SimpleImputer_ is appropriate for replacing missing numeric values, while our explicit benchmark-availability flags retain the business meaning of missing historical benchmarks.


In [0]:
# ============================================================
# Isolation Forest pipeline factory
# ============================================================

def build_isolation_forest_pipeline():

    return Pipeline(
        steps=[
            (
                "imputer",

                SimpleImputer(
                    strategy="median",
                    keep_empty_features=True
                )
            ),

            (
                "isolation_forest",

                IsolationForest(
                    n_estimators=N_ESTIMATORS,
                    max_samples=MAX_SAMPLES,
                    contamination="auto",
                    max_features=MAX_FEATURES,
                    bootstrap=BOOTSTRAP,
                    n_jobs=-1,
                    random_state=RANDOM_STATE
                )
            )
        ]
    )


print(
    "Isolation Forest pipeline factory loaded."
)

Isolation Forest pipeline factory loaded.


**Anomaly score helper functions**

In [0]:
# ============================================================
# Anomaly score helpers
#
# sklearn IsolationForest:
# higher decision_function = more normal
#
# Our convention:
# higher RawAnomalyScore = more anomalous
# higher PricingAnomalyScore = more anomalous
# ============================================================

def calculate_raw_anomaly_score(
    model,
    X
):

    return (
        -1.0
        *
        model.decision_function(
            X
        )
    )


def calculate_reference_percentile_score(
    reference_raw_scores,
    target_raw_scores
):

    reference_sorted = (
        np.sort(
            np.asarray(
                reference_raw_scores,
                dtype=float
            )
        )
    )


    target_array = (
        np.asarray(
            target_raw_scores,
            dtype=float
        )
    )


    percentile_scores = (
        np.searchsorted(
            reference_sorted,
            target_array,
            side="right"
        )
        /
        len(
            reference_sorted
        )
        *
        100.0
    )


    return np.clip(
        percentile_scores,
        0.0,
        100.0
    )


def calculate_raw_threshold(
    reference_raw_scores,
    review_rate
):

    return float(
        np.quantile(
            reference_raw_scores,
            1.0
            -
            review_rate
        )
    )


print(
    "Anomaly score helpers loaded."
)

Anomaly score helpers loaded.


**Train development model**

In [0]:
# ============================================================
# Train development Isolation Forest
#
# Fit only on 2023-2024.
# 2025 remains untouched.
# ============================================================

development_pipeline = (
    build_isolation_forest_pipeline()
)


development_pipeline.fit(
    X_development
)


development_raw_scores = (
    calculate_raw_anomaly_score(
        development_pipeline,
        X_development
    )
)


DEVELOPMENT_RAW_THRESHOLD = (
    calculate_raw_threshold(
        development_raw_scores,
        ANOMALY_REVIEW_RATE
    )
)


development_anomaly_scores = (
    calculate_reference_percentile_score(
        development_raw_scores,
        development_raw_scores
    )
)


development_flags = (
    development_raw_scores
    >=
    DEVELOPMENT_RAW_THRESHOLD
).astype(int)


print(
    "Development Isolation Forest trained."
)

print(
    "Development raw threshold:",
    round(
        DEVELOPMENT_RAW_THRESHOLD,
        6
    )
)

print(
    "Development anomaly rate:",
    f"{development_flags.mean():.2%}"
)

Development Isolation Forest trained.
Development raw threshold: 0.017938
Development anomaly rate: 5.00%


**Score untouched 2025 temporal diagnostic**

In [0]:
# ============================================================
# Score untouched 2025 temporal diagnostic population
# ============================================================

validation_raw_scores = (
    calculate_raw_anomaly_score(
        development_pipeline,
        X_validation
    )
)


validation_anomaly_scores = (
    calculate_reference_percentile_score(
        development_raw_scores,
        validation_raw_scores
    )
)


validation_flags = (
    validation_raw_scores
    >=
    DEVELOPMENT_RAW_THRESHOLD
).astype(int)


validation_proxy = (
    validation_pd[
        "RuleBasedExtremePriceFlag"
    ]
    .fillna(0)
    .astype(int)
    .to_numpy()
)


validation_contract_exception = (
    validation_pd[
        "PriceComplianceExceptionFlag"
    ]
    .fillna(0)
    .astype(int)
    .to_numpy()
)


print(
    "2025 temporal scoring completed."
)

print(
    "2025 predicted anomaly rate:",
    f"{validation_flags.mean():.2%}"
)

print(
    "2025 rule-based extreme-price rate:",
    f"{validation_proxy.mean():.2%}"
)

2025 temporal scoring completed.
2025 predicted anomaly rate: 5.79%
2025 rule-based extreme-price rate: 8.40%


**Evaluate 2025 against diagnostic proxy**

In [0]:
# ============================================================
# Temporal diagnostic metrics
#
# RuleBasedExtremePriceFlag is NOT ground truth.
#
# These metrics only tell us whether high model anomaly
# scores tend to enrich transparent extreme-price rules.
# ============================================================

validation_proxy_prevalence = float(
    validation_proxy.mean()
)


validation_roc_auc = float(
    roc_auc_score(
        validation_proxy,
        validation_anomaly_scores
    )
)


validation_pr_auc = float(
    average_precision_score(
        validation_proxy,
        validation_anomaly_scores
    )
)


validation_precision = float(
    precision_score(
        validation_proxy,
        validation_flags,
        zero_division=0
    )
)


validation_recall = float(
    recall_score(
        validation_proxy,
        validation_flags,
        zero_division=0
    )
)


validation_f1 = float(
    f1_score(
        validation_proxy,
        validation_flags,
        zero_division=0
    )
)


(
    validation_tn,
    validation_fp,
    validation_fn,
    validation_tp
) = (
    confusion_matrix(
        validation_proxy,
        validation_flags,
        labels=[
            0,
            1
        ]
    )
    .ravel()
)


validation_precision_lift = (
    (
        validation_precision
        /
        validation_proxy_prevalence
    )
    if validation_proxy_prevalence > 0
    else np.nan
)


print(
    "2025 TEMPORAL DIAGNOSTIC"
)

print(
    "Proxy prevalence:",
    f"{validation_proxy_prevalence:.2%}"
)

print(
    "Predicted anomaly rate:",
    f"{validation_flags.mean():.2%}"
)

print(
    "ROC-AUC vs diagnostic proxy:",
    round(
        validation_roc_auc,
        4
    )
)

print(
    "PR-AUC vs diagnostic proxy:",
    round(
        validation_pr_auc,
        4
    )
)

print(
    "Precision vs diagnostic proxy:",
    f"{validation_precision:.2%}"
)

print(
    "Recall vs diagnostic proxy:",
    f"{validation_recall:.2%}"
)

print(
    "F1 vs diagnostic proxy:",
    round(
        validation_f1,
        4
    )
)

print(
    "Precision lift vs proxy baseline:",
    round(
        validation_precision_lift,
        2
    )
)

print(
    "\nDiagnostic confusion matrix:"
)

print(
    "TN:",
    validation_tn
)

print(
    "FP:",
    validation_fp
)

print(
    "FN:",
    validation_fn
)

print(
    "TP:",
    validation_tp
)

2025 TEMPORAL DIAGNOSTIC
Proxy prevalence: 8.40%
Predicted anomaly rate: 5.79%
ROC-AUC vs diagnostic proxy: 0.7958
PR-AUC vs diagnostic proxy: 0.2857
Precision vs diagnostic proxy: 35.64%
Recall vs diagnostic proxy: 24.58%
F1 vs diagnostic proxy: 0.2909
Precision lift vs proxy baseline: 4.24

Diagnostic confusion matrix:
TN: 16643
FP: 706
FN: 1200
TP: 391


**Contract-exception enrichment diagnostic**

In [0]:
# ============================================================
# Contract pricing exception enrichment
#
# This is another independent business diagnostic.
# ============================================================

overall_contract_exception_rate = float(
    validation_contract_exception.mean()
)


if validation_flags.sum() > 0:

    anomaly_contract_exception_rate = float(
        validation_contract_exception[
            validation_flags == 1
        ]
        .mean()
    )

else:

    anomaly_contract_exception_rate = np.nan


contract_exception_lift = (
    (
        anomaly_contract_exception_rate
        /
        overall_contract_exception_rate
    )
    if (
        overall_contract_exception_rate > 0
        and
        math.isfinite(
            anomaly_contract_exception_rate
        )
    )
    else np.nan
)


print(
    "2025 CONTRACT-EXCEPTION DIAGNOSTIC"
)

print(
    "Overall price-compliance exception rate:",
    f"{overall_contract_exception_rate:.2%}"
)

print(
    "Exception rate among model anomalies:",
    (
        f"{anomaly_contract_exception_rate:.2%}"
        if math.isfinite(
            anomaly_contract_exception_rate
        )
        else "N/A"
    )
)

print(
    "Exception-rate lift:",
    (
        round(
            contract_exception_lift,
            2
        )
        if math.isfinite(
            contract_exception_lift
        )
        else "N/A"
    )
)

2025 CONTRACT-EXCEPTION DIAGNOSTIC
Overall price-compliance exception rate: 1.38%
Exception rate among model anomalies: 11.03%
Exception-rate lift: 7.97


**Inspect 2025 anomaly score distribution**

In [0]:
# ============================================================
# Inspect 2025 anomaly-score distribution
# ============================================================

validation_score_summary_pd = (
    pd.Series(
        validation_anomaly_scores
    )
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


display(
    validation_score_summary_pd
)

count    18940.000000
mean        51.250083
std         29.136731
min          0.017723
50%         50.248128
75%         78.345297
90%         91.794054
95%         95.640037
99%         99.020781
max         99.946830
dtype: float64

**MLflow model logging helper**

In [0]:
# ============================================================
# MLflow sklearn logging compatibility helper
# ============================================================

def log_sklearn_model_compat(
    model,
    X_example,
    model_name="model"
):

    input_example = (
        X_example
        .head(10)
        .copy()
    )


    output_example = (
        model.predict(
            input_example
        )
    )


    signature = (
        infer_signature(
            input_example,
            output_example
        )
    )


    try:

        model_info = (
            mlflow.sklearn.log_model(
                sk_model=model,
                name=model_name,
                signature=signature,
                input_example=input_example
            )
        )

    except TypeError:

        model_info = (
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path=model_name,
                signature=signature,
                input_example=input_example
            )
        )


    return model_info


print(
    "MLflow sklearn logging helper loaded."
)

MLflow sklearn logging helper loaded.


**Log temporal diagnostic model**

In [0]:
# ============================================================
# Log development / temporal-diagnostic model
# ============================================================

if mlflow.active_run() is not None:

    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        "pricing_anomaly_"
        "isolation_forest_"
        "temporal_validation"
    )
) as temporal_run:

    mlflow.set_tags({
        "project":
            (
                "Enterprise Procurement "
                "Intelligence Platform"
            ),

        "model_family":
            MODEL_FAMILY,

        "model_name":
            MODEL_NAME,

        "model_stage":
            "temporal_validation",

        "model_status":
            MODEL_STATUS,

        "development_years":
            "2023-2024",

        "validation_year":
            "2025",

        "diagnostic_proxy":
            "RuleBasedExtremePriceFlag",

        "ground_truth_available":
            "false"
    })


    mlflow.log_params({
        "n_estimators":
            N_ESTIMATORS,

        "max_samples":
            MAX_SAMPLES,

        "max_features":
            MAX_FEATURES,

        "bootstrap":
            BOOTSTRAP,

        "contamination":
            "auto",

        "random_state":
            RANDOM_STATE,

        "review_rate":
            ANOMALY_REVIEW_RATE,

        "raw_anomaly_threshold":
            DEVELOPMENT_RAW_THRESHOLD,

        "feature_count":
            len(
                PRICING_MODEL_FEATURES
            )
    })


    mlflow.log_metrics({
        "validation_proxy_prevalence":
            validation_proxy_prevalence,

        "validation_anomaly_rate":
            float(
                validation_flags.mean()
            ),

        "validation_proxy_roc_auc":
            validation_roc_auc,

        "validation_proxy_pr_auc":
            validation_pr_auc,

        "validation_proxy_precision":
            validation_precision,

        "validation_proxy_recall":
            validation_recall,

        "validation_proxy_f1":
            validation_f1,

        "validation_precision_lift":
            float(
                validation_precision_lift
            ),

        "overall_contract_exception_rate":
            overall_contract_exception_rate,

        "anomaly_contract_exception_rate":
            float(
                anomaly_contract_exception_rate
            )
            if math.isfinite(
                anomaly_contract_exception_rate
            )
            else 0.0
    })


    temporal_model_info = (
        log_sklearn_model_compat(
            development_pipeline,
            X_development,
            model_name="model"
        )
    )


    TEMPORAL_RUN_ID = (
        temporal_run.info.run_id
    )


    TEMPORAL_MODEL_URI = (
        temporal_model_info.model_uri
    )


print(
    "Temporal diagnostic model logged."
)

print(
    "Run ID:",
    TEMPORAL_RUN_ID
)

print(
    "Model URI:",
    TEMPORAL_MODEL_URI
)

🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/2316106049324850/models/m-f9b0b9efc7824ca787c356c06ff05ffd?o=7405606393632123


Temporal diagnostic model logged.
Run ID: ff5c5e2e20f04b4f97a5bfc70a0d75ef
Model URI: models:/m-f9b0b9efc7824ca787c356c06ff05ffd


**Build persisted 2025 validation output**

In [0]:
# ============================================================
# Build 2025 temporal validation output
# ============================================================

validation_scores_pd = pd.DataFrame({
    "POItemID":
        validation_pd[
            "POItemID"
        ].astype(str),

    "RawAnomalyScore":
        validation_raw_scores.astype(float),

    "PricingAnomalyScore":
        validation_anomaly_scores.astype(float),

    "PricingAnomalyFlag":
        validation_flags.astype(int)
})


validation_scores_spark_df = (
    spark.createDataFrame(
        validation_scores_pd
    )
)


validation_predictions_df = (
    validation_df

    .join(
        validation_scores_spark_df,

        on="POItemID",

        how="inner"
    )

    .withColumn(
        "ModelName",
        F.lit(
            MODEL_NAME
        )
    )

    .withColumn(
        "ModelRunID",
        F.lit(
            TEMPORAL_RUN_ID
        )
    )

    .withColumn(
        "ModelStage",
        F.lit(
            "TemporalValidation"
        )
    )

    .withColumn(
        "ModelStatus",
        F.lit(
            MODEL_STATUS
        )
    )

    .withColumn(
        "AnomalyReviewRate",
        F.lit(
            ANOMALY_REVIEW_RATE
        )
    )

    .withColumn(
        "RawAnomalyThreshold",
        F.lit(
            DEVELOPMENT_RAW_THRESHOLD
        )
    )

    .withColumn(
        "PredictionTimestampUTC",
        F.current_timestamp()
    )
)


print(
    "Validation output rows:",
    f"{validation_predictions_df.count():,}"
)

Validation output rows: 18,940


**DB_05 temporal quality gate**

In [0]:
# ============================================================
# Temporal anomaly-model quality gate
#
# Because no reviewed anomaly label exists, the gate focuses
# on technical validity and non-degenerate anomaly behavior.
#
# Proxy metrics are reported but are not treated as production
# accuracy.
# ============================================================

validation_anomaly_rate = float(
    validation_flags.mean()
)


validation_unique_flag_count = int(
    np.unique(
        validation_flags
    ).size
)


validation_null_score_count = int(
    np.isnan(
        validation_anomaly_scores
    ).sum()
)


validation_invalid_score_count = int(
    (
        (
            validation_anomaly_scores
            < 0
        )
        |
        (
            validation_anomaly_scores
            > 100
        )
    )
    .sum()
)


quality_checks = [

    (
        "Development population contains rows",
        development_count > 0
    ),

    (
        "2025 temporal population contains rows",
        validation_count > 0
    ),

    (
        "Development threshold is finite",
        math.isfinite(
            DEVELOPMENT_RAW_THRESHOLD
        )
    ),

    (
        "2025 anomaly score contains no nulls",
        validation_null_score_count == 0
    ),

    (
        "2025 anomaly score is between 0 and 100",
        validation_invalid_score_count == 0
    ),

    (
        "2025 model predicts both classes",
        validation_unique_flag_count == 2
    ),

    (
        "2025 anomaly rate is not degenerate",
        (
            validation_anomaly_rate
            >= 0.005
            and
            validation_anomaly_rate
            <= 0.20
        )
    ),

    (
        "Diagnostic ROC-AUC is finite",
        math.isfinite(
            validation_roc_auc
        )
    ),

    (
        "Diagnostic PR-AUC is finite",
        math.isfinite(
            validation_pr_auc
        )
    )
]


failed_checks = []


for (
    check_name,
    passed
) in quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_checks.append(
            check_name
        )


print(
    "\nTEMPORAL DIAGNOSTICS"
)

print(
    "2025 anomaly rate:",
    f"{validation_anomaly_rate:.2%}"
)

print(
    "Proxy baseline:",
    f"{validation_proxy_prevalence:.2%}"
)

print(
    "Proxy PR-AUC:",
    round(
        validation_pr_auc,
        4
    )
)

print(
    "Proxy precision:",
    f"{validation_precision:.2%}"
)

print(
    "Precision lift:",
    round(
        validation_precision_lift,
        2
    )
)


if (
    validation_pr_auc
    <=
    validation_proxy_prevalence
):

    print(
        "WARNING | Model anomaly ranking does not "
        "outperform the diagnostic proxy prevalence "
        "baseline on PR-AUC."
    )


if failed_checks:

    raise ValueError(
        "DB_05 temporal quality gate FAILED: "
        +
        "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_05 TEMPORAL QUALITY GATE PASSED."
)

PASS | Development population contains rows
PASS | 2025 temporal population contains rows
PASS | Development threshold is finite
PASS | 2025 anomaly score contains no nulls
PASS | 2025 anomaly score is between 0 and 100
PASS | 2025 model predicts both classes
PASS | 2025 anomaly rate is not degenerate
PASS | Diagnostic ROC-AUC is finite
PASS | Diagnostic PR-AUC is finite

TEMPORAL DIAGNOSTICS
2025 anomaly rate: 5.79%
Proxy baseline: 8.40%
Proxy PR-AUC: 0.2857
Proxy precision: 35.64%
Precision lift: 4.24

DB_05 TEMPORAL QUALITY GATE PASSED.


**Retrain production candidate on 2023-2025**

In [0]:
# ============================================================
# Retrain production candidate
#
# After completing the untouched 2025 diagnostic,
# production training may now use 2023-2025.
# ============================================================

production_pipeline = (
    build_isolation_forest_pipeline()
)


production_pipeline.fit(
    X_production
)


production_training_raw_scores = (
    calculate_raw_anomaly_score(
        production_pipeline,
        X_production
    )
)


PRODUCTION_RAW_THRESHOLD = (
    calculate_raw_threshold(
        production_training_raw_scores,
        ANOMALY_REVIEW_RATE
    )
)


production_training_flags = (
    production_training_raw_scores
    >=
    PRODUCTION_RAW_THRESHOLD
).astype(int)


print(
    "Production Isolation Forest trained."
)

print(
    "Production training rows:",
    f"{len(X_production):,}"
)

print(
    "Production raw threshold:",
    round(
        PRODUCTION_RAW_THRESHOLD,
        6
    )
)

print(
    "Historical production anomaly rate:",
    f"{production_training_flags.mean():.2%}"
)

Production Isolation Forest trained.
Production training rows: 41,509
Production raw threshold: 0.014884
Historical production anomaly rate: 5.00%


**Score 2026**

In [0]:
# ============================================================
# Score 2026 pricing population
# ============================================================

scoring_raw_scores = (
    calculate_raw_anomaly_score(
        production_pipeline,
        X_scoring
    )
)


scoring_anomaly_scores = (
    calculate_reference_percentile_score(
        production_training_raw_scores,
        scoring_raw_scores
    )
)


scoring_flags = (
    scoring_raw_scores
    >=
    PRODUCTION_RAW_THRESHOLD
).astype(int)


scoring_anomaly_rate = float(
    scoring_flags.mean()
)


print(
    "2026 PO items scored:",
    f"{len(scoring_flags):,}"
)

print(
    "2026 pricing anomalies:",
    f"{int(scoring_flags.sum()):,}"
)

print(
    "2026 anomaly rate:",
    f"{scoring_anomaly_rate:.2%}"
)

2026 PO items scored: 21,752
2026 pricing anomalies: 1,216
2026 anomaly rate: 5.59%


**Inspect 2026 score distribution**

In [0]:
# ============================================================
# 2026 anomaly-score distribution
# ============================================================

scoring_score_summary_pd = (
    pd.Series(
        scoring_anomaly_scores
    )
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


display(
    scoring_score_summary_pd
)

count    21752.000000
mean        52.816111
std         29.742561
min          0.000000
50%         56.692525
75%         78.272784
90%         91.196849
95%         95.594811
99%         99.136308
max         99.980727
dtype: float64

**Log production model**

In [0]:
# ============================================================
# Log production pricing anomaly model
# ============================================================

if mlflow.active_run() is not None:

    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        "pricing_anomaly_"
        "isolation_forest_"
        "production_candidate"
    )
) as production_run:

    mlflow.set_tags({
        "project":
            (
                "Enterprise Procurement "
                "Intelligence Platform"
            ),

        "model_family":
            MODEL_FAMILY,

        "model_name":
            MODEL_NAME,

        "model_stage":
            "production_candidate",

        "model_status":
            MODEL_STATUS,

        "training_years":
            "2023-2025",

        "scoring_year":
            "2026",

        "ground_truth_available":
            "false"
    })


    mlflow.log_params({
        "n_estimators":
            N_ESTIMATORS,

        "max_samples":
            MAX_SAMPLES,

        "max_features":
            MAX_FEATURES,

        "bootstrap":
            BOOTSTRAP,

        "contamination":
            "auto",

        "random_state":
            RANDOM_STATE,

        "review_rate":
            ANOMALY_REVIEW_RATE,

        "raw_anomaly_threshold":
            PRODUCTION_RAW_THRESHOLD,

        "anomaly_score_cutoff":
            ANOMALY_SCORE_CUTOFF,

        "feature_count":
            len(
                PRICING_MODEL_FEATURES
            )
    })


    # --------------------------------------------------------
    # Historical validation metrics are reference metrics.
    # The production model itself was subsequently retrained
    # using the 2025 population.
    # --------------------------------------------------------

    mlflow.log_metrics({
        "reference_2025_proxy_roc_auc":
            validation_roc_auc,

        "reference_2025_proxy_pr_auc":
            validation_pr_auc,

        "reference_2025_proxy_precision":
            validation_precision,

        "reference_2025_proxy_recall":
            validation_recall,

        "reference_2025_precision_lift":
            validation_precision_lift,

        "2026_anomaly_rate":
            scoring_anomaly_rate
    })


    production_model_info = (
        log_sklearn_model_compat(
            production_pipeline,
            X_production,
            model_name="model"
        )
    )


    PRODUCTION_RUN_ID = (
        production_run.info.run_id
    )


    PRODUCTION_MODEL_URI = (
        production_model_info.model_uri
    )


print(
    "Production pricing anomaly model logged."
)

print(
    "Run ID:",
    PRODUCTION_RUN_ID
)

print(
    "Model URI:",
    PRODUCTION_MODEL_URI
)

🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/2316106049324850/models/m-ccfe1dfd66654573b4a218b2f5a25887?o=7405606393632123


Production pricing anomaly model logged.
Run ID: fdb24ff14d3648bd8e611c6e4792712f
Model URI: models:/m-ccfe1dfd66654573b4a218b2f5a25887


**Build 2026 scoring output**

In [0]:
# ============================================================
# Build 2026 pricing anomaly prediction output
# ============================================================

scoring_scores_pd = pd.DataFrame({
    "POItemID":
        scoring_pd[
            "POItemID"
        ].astype(str),

    "RawAnomalyScore":
        scoring_raw_scores.astype(float),

    "PricingAnomalyScore":
        scoring_anomaly_scores.astype(float),

    "PricingAnomalyFlag":
        scoring_flags.astype(int)
})


scoring_scores_spark_df = (
    spark.createDataFrame(
        scoring_scores_pd
    )
)


scoring_predictions_df = (
    pricing_scoring_df

    .join(
        scoring_scores_spark_df,

        on="POItemID",

        how="inner"
    )

    .withColumn(
        "ModelName",
        F.lit(
            MODEL_NAME
        )
    )

    .withColumn(
        "ModelRunID",
        F.lit(
            PRODUCTION_RUN_ID
        )
    )

    .withColumn(
        "ModelStage",
        F.lit(
            "ProductionCandidate"
        )
    )

    .withColumn(
        "ModelStatus",
        F.lit(
            MODEL_STATUS
        )
    )

    .withColumn(
        "AnomalyReviewRate",
        F.lit(
            ANOMALY_REVIEW_RATE
        )
    )

    .withColumn(
        "RawAnomalyThreshold",
        F.lit(
            PRODUCTION_RAW_THRESHOLD
        )
    )

    .withColumn(
        "AnomalyScoreCutoff",
        F.lit(
            ANOMALY_SCORE_CUTOFF
        )
    )

    .withColumn(
        "PredictionTimestampUTC",
        F.current_timestamp()
    )
)


print(
    "2026 scoring output rows:",
    f"{scoring_predictions_df.count():,}"
)

2026 scoring output rows: 21,752


**Inspect highest 2026 pricing anomalies**

In [0]:
# ============================================================
# Inspect highest-scoring 2026 pricing anomalies
# ============================================================

display(
    scoring_predictions_df

    .select(
        "POItemID",
        "POID",
        "OrderDate",

        "SupplierID",
        "SupplierName",

        "MaterialID",
        "MaterialName",

        "CategoryID",
        "CategoryName",

        "ContractID",

        "UnitPriceEUR",
        "LineAmountEUR",

        "ContractPriceVariancePct",

        "PriceVsMaterialHistoricalAvgPct",
        "PriceVsSupplierMaterialHistoricalAvgPct",
        "PriceVsCategoryHistoricalAvgPct",

        "MaterialPriceZScore",
        "SupplierMaterialPriceZScore",
        "CategoryPriceZScore",

        "BenchmarkCoverageCount",

        "RawAnomalyScore",
        "PricingAnomalyScore",
        "PricingAnomalyFlag",

        "RuleBasedExtremePriceFlag",
        "PriceComplianceExceptionFlag"
    )

    .orderBy(
        F.desc(
            "PricingAnomalyScore"
        ),
        F.desc(
            "LineAmountEUR"
        )
    )

    .limit(50)
)

POItemID,POID,OrderDate,SupplierID,SupplierName,MaterialID,MaterialName,CategoryID,CategoryName,ContractID,UnitPriceEUR,LineAmountEUR,ContractPriceVariancePct,PriceVsMaterialHistoricalAvgPct,PriceVsSupplierMaterialHistoricalAvgPct,PriceVsCategoryHistoricalAvgPct,MaterialPriceZScore,SupplierMaterialPriceZScore,CategoryPriceZScore,BenchmarkCoverageCount,RawAnomalyScore,PricingAnomalyScore,PricingAnomalyFlag,RuleBasedExtremePriceFlag,PriceComplianceExceptionFlag
4500002893-00010,4500002893,2026-01-06,SUP000401,BluePeak Advanced Engineering B.V.,MAT001109,Mechanical Component 00114,CAT003,Mechanical Components,null,22005.0586,264060.7,null,1720.607352167925,6935.7020080393195,2458.967411622948,5.046455461483082,2464.2176942589904,6.989155278773666,2,0.22177333700770452,99.98072707123757,1,1,0
4500000166-00020,4500000166,2026-03-17,SUP000045,Aurora Dynamic Equipment Group,MAT000699,Temporary Labor 00014,CAT015,Temporary Labor,null,4547.7088,51511.89,null,1809.2155857047785,4777.347588321786,2877.513331828333,9.887282544404345,484.1615778155515,21.268077764334972,3,0.22411614145681513,99.98072707123757,1,1,0
4500017328-00010,4500017328,2026-07-15,SUP000452,Falcon Advanced Supply AG,MAT001294,Freight Movement 00033,CAT011,Logistics and Freight,null,25303.2,278335.2,null,788.0755818108805,3194.312886295048,1868.7624766578238,6.618199764711784,2204.6143805853662,9.787935306274214,3,0.21849697346862307,99.97831795514226,1,1,0
4500001357-00010,4500001357,2026-05-13,SUP000290,"Pioneer Sustainable Components Co., Ltd.",MAT000744,Warehousing Service 00022,CAT012,Warehousing,null,55182.823,55182.82,null,1101.3046544030071,1067.7888547684,3814.6424868585677,6.7300809461772255,5.654124559179187,14.676111011239275,3,0.22024666899933054,99.97831795514226,1,1,0
4500017981-00020,4500017981,2026-02-10,SUP000409,Keystone Integrated Manufacturing Ltd.,MAT000857,MRO Item 00018,CAT008,"Maintenance, Repair and Operations",null,9794.3269,19588.65,null,363.31491902015955,8192.918825415076,2344.553547160304,1.9517049137272118,10107.680486010187,7.269325168023044,3,0.22009726618991032,99.97831795514226,1,1,0
4500005958-00010,4500005958,2026-07-20,SUP000479,Frontier Global Electronics Ltd.,MAT001396,Steel and Alloy Material 00116,CAT001,Raw Materials - Metals,null,14065.3984,18721.04,null,184.31687177483155,18850.077850526093,4353.432306686945,1.392392359822145,2818.250754986302,11.077688494525145,3,0.21854964169306568,99.97831795514226,1,1,0
4500009917-00070,4500009917,2026-04-21,SUP000034,Ironwood Advanced Services Group,MAT000008,Energy or Utility 00001,CAT017,Energy and Utilities,null,182820.3,182820.3,null,292.32975202699134,25411.73128496506,2600.338170694954,1.6782961617104024,2660.583066341628,4.891614916000384,3,0.21400397410620398,99.96145414247512,1,1,0
4500016654-00050,4500016654,2026-04-01,SUP000208,BluePeak Sustainable Technology Ltd.,MAT000059,Marketing Service 00001,CAT018,Marketing and Communications,null,224301.23,224301.23,null,628.0361900406829,3126.525333644697,1526.852259799701,2.7754320213736965,1435.825514203313,6.68391845367553,3,0.21248536581179944,99.95663591028452,1,1,0
4500006893-00020,4500006893,2026-07-20,SUP000169,Terra Advanced Systems Corp.,MAT000625,IT Product or Service 00013,CAT013,Information Technology,null,205838.9,205838.9,null,433.60802427429485,3926.1459125011947,1870.523975433217,2.2034637236347288,1208.4605974755534,6.603024444944299,3,0.21156908308061573,99.95422679418921,1,1,0
4500009257-00010,4500009257,2026-02-26,SUP000455,Cobalt Sustainable Resources AG,MAT001245,MRO Item 00029,CAT008,"Maintenance, Repair and Operations",null,9990.0486,9990.05,null,412.2037104514834,9256.20851187933,2387.7147849431144,2.148511585429642,999.9636186522997,7.4083639699504715,3,0.2114187370594245,99.95422679418921,1,1,0


**Build model metadata**

In [0]:
# ============================================================
# Build Pricing Anomaly model metadata
# ============================================================

metadata_rows = [
    (
        MODEL_FAMILY,
        MODEL_NAME,
        MODEL_STATUS,

        PRODUCTION_RUN_ID,
        PRODUCTION_MODEL_URI,

        DEVELOPMENT_START_YEAR,
        DEVELOPMENT_END_YEAR,

        TEMPORAL_VALIDATION_YEAR,
        SCORING_YEAR,

        N_ESTIMATORS,
        str(
            MAX_SAMPLES
        ),
        MAX_FEATURES,

        ANOMALY_REVIEW_RATE,
        ANOMALY_SCORE_CUTOFF,
        PRODUCTION_RAW_THRESHOLD,

        validation_proxy_prevalence,
        validation_roc_auc,
        validation_pr_auc,
        validation_precision,
        validation_recall,
        validation_f1,
        validation_precision_lift,

        scoring_anomaly_rate,

        production_training_count,
        scoring_row_count,

        len(
            PRICING_MODEL_FEATURES
        )
    )
]


metadata_columns = [
    "ModelFamily",
    "ModelName",
    "ModelStatus",

    "ModelRunID",
    "ModelURI",

    "DevelopmentStartYear",
    "DevelopmentEndYear",

    "TemporalValidationYear",
    "ScoringYear",

    "NEstimators",
    "MaxSamples",
    "MaxFeatures",

    "AnomalyReviewRate",
    "AnomalyScoreCutoff",
    "RawAnomalyThreshold",

    "ValidationProxyPrevalence",
    "ValidationProxyROCAUC",
    "ValidationProxyPRAUC",
    "ValidationProxyPrecision",
    "ValidationProxyRecall",
    "ValidationProxyF1",
    "ValidationPrecisionLift",

    "ScoringAnomalyRate",

    "ProductionTrainingRowCount",
    "ScoringRowCount",

    "FeatureCount"
]


model_metadata_df = (
    spark.createDataFrame(
        metadata_rows,
        metadata_columns
    )

    .withColumn(
        "GroundTruthAvailableFlag",
        F.lit(0)
    )

    .withColumn(
        "ValidationProxyName",
        F.lit(
            "RuleBasedExtremePriceFlag"
        )
    )

    .withColumn(
        "CreatedTimestampUTC",
        F.current_timestamp()
    )
)


display(
    model_metadata_df
)

ModelFamily,ModelName,ModelStatus,ModelRunID,ModelURI,DevelopmentStartYear,DevelopmentEndYear,TemporalValidationYear,ScoringYear,NEstimators,MaxSamples,MaxFeatures,AnomalyReviewRate,AnomalyScoreCutoff,RawAnomalyThreshold,ValidationProxyPrevalence,ValidationProxyROCAUC,ValidationProxyPRAUC,ValidationProxyPrecision,ValidationProxyRecall,ValidationProxyF1,ValidationPrecisionLift,ScoringAnomalyRate,ProductionTrainingRowCount,ScoringRowCount,FeatureCount,GroundTruthAvailableFlag,ValidationProxyName,CreatedTimestampUTC
Pricing Anomaly Detection,IsolationForest,Experimental,fdb24ff14d3648bd8e611c6e4792712f,models:/m-ccfe1dfd66654573b4a218b2f5a25887,2023,2024,2025,2026,300,auto,1.0,0.05,95.0,0.014883543396063657,0.08400211193241816,0.7958331598873847,0.2857244147220301,0.35642661804922515,0.245757385292269,0.29092261904761907,4.243067344973177,0.05590290547995587,41509,21752,19,0,RuleBasedExtremePriceFlag,2026-08-13T07:51:46.033598Z


**Final DB_05 scoring quality gates**

In [0]:
# ============================================================
# Final DB_05 scoring quality gates
# ============================================================

scoring_unique_po_item_count = (
    scoring_predictions_df

    .select(
        "POItemID"
    )

    .distinct()

    .count()
)


scoring_prediction_count = (
    scoring_predictions_df.count()
)


scoring_null_score_count = (
    scoring_predictions_df

    .filter(
        F.col(
            "PricingAnomalyScore"
        ).isNull()
    )

    .count()
)


scoring_invalid_score_count = (
    scoring_predictions_df

    .filter(
        (
            F.col(
                "PricingAnomalyScore"
            )
            < 0
        )
        |
        (
            F.col(
                "PricingAnomalyScore"
            )
            > 100
        )
    )

    .count()
)


scoring_distinct_flag_count = (
    scoring_predictions_df

    .select(
        "PricingAnomalyFlag"
    )

    .distinct()

    .count()
)


final_quality_checks = [

    (
        "2026 prediction row count matches DB_04 scoring store",
        scoring_prediction_count == scoring_row_count
    ),

    (
        "2026 prediction grain is unique by POItemID",
        scoring_prediction_count
        ==
        scoring_unique_po_item_count
    ),

    (
        "2026 anomaly score contains no nulls",
        scoring_null_score_count == 0
    ),

    (
        "2026 anomaly score is between 0 and 100",
        scoring_invalid_score_count == 0
    ),

    (
        "2026 scoring predicts both classes",
        scoring_distinct_flag_count == 2
    ),

    (
        "2026 anomaly rate is non-degenerate",
        (
            scoring_anomaly_rate >= 0.005
            and
            scoring_anomaly_rate <= 0.20
        )
    ),

    (
        "Production model run ID exists",
        bool(
            PRODUCTION_RUN_ID
        )
    ),

    (
        "Production model URI exists",
        bool(
            PRODUCTION_MODEL_URI
        )
    )
]


failed_final_checks = []


for (
    check_name,
    passed
) in final_quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_final_checks.append(
            check_name
        )


print(
    "\n2026 pricing anomaly rate:",
    f"{scoring_anomaly_rate:.2%}"
)


if failed_final_checks:

    raise ValueError(
        "DB_05 final quality gate FAILED: "
        +
        "; ".join(
            failed_final_checks
        )
    )


print(
    "\nDB_05 MODEL QUALITY GATE PASSED."
)

PASS | 2026 prediction row count matches DB_04 scoring store
PASS | 2026 prediction grain is unique by POItemID
PASS | 2026 anomaly score contains no nulls
PASS | 2026 anomaly score is between 0 and 100
PASS | 2026 scoring predicts both classes
PASS | 2026 anomaly rate is non-degenerate
PASS | Production model run ID exists
PASS | Production model URI exists

2026 pricing anomaly rate: 5.59%

DB_05 MODEL QUALITY GATE PASSED.


**Persist DB_05 outputs**

In [0]:
# ============================================================
# Persist DB_05 outputs to OneLake
# ============================================================

(
    validation_predictions_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        VALIDATION_PREDICTIONS_PATH
    )
)


(
    scoring_predictions_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        SCORING_PREDICTIONS_PATH
    )
)


(
    model_metadata_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        MODEL_METADATA_PATH
    )
)


print(
    "DB_05 outputs persisted successfully."
)

DB_05 outputs persisted successfully.


**Persistence validation**

In [0]:
# ============================================================
# Validate persisted DB_05 outputs
# ============================================================

persisted_validation_df = (
    spark.read
    .format("delta")
    .load(
        VALIDATION_PREDICTIONS_PATH
    )
)


persisted_scoring_df = (
    spark.read
    .format("delta")
    .load(
        SCORING_PREDICTIONS_PATH
    )
)


persisted_metadata_df = (
    spark.read
    .format("delta")
    .load(
        MODEL_METADATA_PATH
    )
)


persisted_validation_count = (
    persisted_validation_df.count()
)


persisted_scoring_count = (
    persisted_scoring_df.count()
)


persisted_metadata_count = (
    persisted_metadata_df.count()
)


if (
    persisted_validation_count
    !=
    validation_count
):

    raise ValueError(
        "DB_05 validation prediction "
        "persistence count mismatch."
    )


if (
    persisted_scoring_count
    !=
    scoring_row_count
):

    raise ValueError(
        "DB_05 scoring prediction "
        "persistence count mismatch."
    )


if persisted_metadata_count != 1:

    raise ValueError(
        "DB_05 model metadata "
        "persistence validation failed."
    )


print(
    "DB_05 persistence validation PASSED."
)

print(
    "2025 validation rows:",
    f"{persisted_validation_count:,}"
)

print(
    "2026 scoring rows:",
    f"{persisted_scoring_count:,}"
)

print(
    "Model metadata rows:",
    persisted_metadata_count
)

print(
    "\nDB_05 PRICING ANOMALY MODEL PASSED."
)

DB_05 persistence validation PASSED.
2025 validation rows: 18,940
2026 scoring rows: 21,752
Model metadata rows: 1

DB_05 PRICING ANOMALY MODEL PASSED.
